# S12b — complete the S12 secondary quantum arms (CPU only, ≈ 1 h, no GPU quota)
In S12, the *patch* version of amplitude encoding crashed (48 patch numbers < 64 amplitudes), so 3 secondary configurations
per dataset never ran (patch amplitude encoding, basis encoding, circuit sizes). **Bug fix: zero-pad 48 → 64** (standard
amplitude encoding). Nothing else changes; all existing S12 results are reused as they are, then the full S12 report is rebuilt.
Primary endpoints Q1–Q3 are not affected (they had already finished).

**Inputs (Add Input):** 1) the **S12 — quantum** notebook output · 2) **S8 replication** output · 3) **S2 baseline-audit** output.
**Accelerator: None (CPU)** · Internet not needed · Save & Run All (Commit). Send back `S12_results_complete.zip`.

In [ ]:
# ===================== CELL 1 — code + inputs, copy the existing S12 results =====================
import os, sys, glob, json, shutil, subprocess, threading, time
from pathlib import Path
OUT = "/kaggle/working/S12"; WD = "/kaggle/working/s12_code"
os.makedirs(WD, exist_ok=True)
FILES = {"qsim.py": r'''# qsim.py - exact state-vector simulator for the poster's Variational Quantum Circuit (VQC), batched, written once for
# both numpy (local tests) and torch (training with autograd). Poster: angle encoding -> L x [Rx Ry Rz per qubit
# (trainable) + entangling CNOT ring] -> measure <Z_i> of every qubit.
import numpy as np

class Backend:
    def __init__(self, lib):
        self.lib = lib; self.torch = lib.__name__ == "torch"
    def cplx(self, re, im):
        return self.lib.complex(re, im) if self.torch else re + 1j * im
    def stack(self, L, dim):
        return self.lib.stack(L, dim) if self.torch else np.stack(L, dim)
    def zeros_like_real(self, x):
        return self.lib.zeros_like(x)
    def index(self, perm):
        return self.lib.as_tensor(perm, dtype=self.lib.long) if self.torch else np.asarray(perm)

def rot(bk, theta, kind):
    """theta: (...,) real -> (...,2,2) complex rotation matrix RX / RY / RZ"""
    L = bk.lib; c, s = L.cos(theta / 2), L.sin(theta / 2); z = bk.zeros_like_real(theta)
    if kind == "x":
        rows = [[bk.cplx(c, z), bk.cplx(z, -s)], [bk.cplx(z, -s), bk.cplx(c, z)]]
    elif kind == "y":
        rows = [[bk.cplx(c, z), bk.cplx(-s, z)], [bk.cplx(s, z), bk.cplx(c, z)]]
    else:
        rows = [[bk.cplx(c, -s), bk.cplx(z, z)], [bk.cplx(z, z), bk.cplx(c, s)]]
    return bk.stack([bk.stack(r, -1) for r in rows], -2)

def apply_1q(bk, state, U, q, n):
    """state (B, 2^n); U (2,2) shared or (B,2,2) per sample; qubit q = most-significant-first (PennyLane wire order)"""
    B = state.shape[0]; s = state.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    s = bk.lib.einsum("ij,akjb->akib", U, s) if U.ndim == 2 else bk.lib.einsum("aij,akjb->akib", U, s)
    return s.reshape(B, 2 ** n)

def cnot_ring_perm(n):
    """basis permutation of CNOT(0,1) CNOT(1,2) ... CNOT(n-1,0) applied in this order (n >= 3; n = 2 -> one CNOT)"""
    pairs = [(i, (i + 1) % n) for i in range(n)] if n > 2 else [(0, 1)]
    idx = np.arange(2 ** n)
    bit = lambda v, q: (v >> (n - 1 - q)) & 1
    for c, t in pairs:                       # new_state[i] = old_state[perm(i)], composed gate by gate
        src = np.array([i ^ (1 << (n - 1 - t)) if bit(i, c) else i for i in range(2 ** n)])
        idx = idx[src]
    return idx

def zsign(n):
    return np.array([[1 - 2 * ((i >> (n - 1 - q)) & 1) for q in range(n)] for i in range(2 ** n)], dtype=np.float32)

def angle_state(bk, angles):
    """product state RY(theta_q)|0> on every wire: (B,n) -> (B,2^n) (exact; identical to applying RY gates)"""
    B, n = angles.shape; L = bk.lib
    st = bk.cplx(L.ones((B, 1)) if bk.torch else np.ones((B, 1)), L.zeros((B, 1)) if bk.torch else np.zeros((B, 1)))
    for q in range(n):
        a = angles[:, q]; v = bk.stack([bk.cplx(L.cos(a / 2), L.zeros_like(a)), bk.cplx(L.sin(a / 2), L.zeros_like(a))], 1)
        st = (st.unsqueeze(-1) * v.unsqueeze(1)).reshape(B, -1) if bk.torch else (st[:, :, None] * v[:, None, :]).reshape(B, -1)
    return st

def vqc_state(bk, angles, W, perm, init=None, reupload=False, entangle=True):
    """angles (B,n): encoding angles; W (L,n,3) trainable (Rx,Ry,Rz) -> final state (B,2^n).
    init      : optional prepared state (B,2^n) (amplitude / basis encoding); default = angle encoding RY(angles)|0>
    reupload  : re-apply the encoding RY(angles) before every layer after the first (data re-uploading)
    entangle  : False removes the CNOT ring (no-entanglement ablation)"""
    n = angles.shape[1]
    st = angle_state(bk, angles) if init is None else init
    for l in range(W.shape[0]):
        if reupload and l > 0:
            Ry = rot(bk, angles, "y")                                                                # (B,n,2,2)
            for q in range(n):
                st = apply_1q(bk, st, Ry[:, q], q, n)
        U = rot(bk, W[l, :, 2], "z") @ rot(bk, W[l, :, 1], "y") @ rot(bk, W[l, :, 0], "x")       # (n,2,2): Rx first, then Ry, Rz
        for q in range(n):
            st = apply_1q(bk, st, U[q], q, n)
        if entangle:
            st = st[:, perm]
    return st

def feature_map_state(bk, angles, reps=2):
    """fixed (non-trainable) entangling feature map for the quantum kernel: reps x [RY(x) on all wires, CNOT ring,
    RZ(x) on all wires] starting from |0...0> (in the spirit of Havlicek et al. 2019)"""
    B, n = angles.shape; perm = cnot_ring_perm(n)
    st = angle_state(bk, angles)
    for r in range(reps):
        if r > 0:
            Ry = rot(bk, angles, "y")
            for q in range(n):
                st = apply_1q(bk, st, Ry[:, q], q, n)
        st = st[:, perm]
        Rz = rot(bk, angles, "z")
        for q in range(n):
            st = apply_1q(bk, st, Rz[:, q], q, n)
    return st

def expz(bk, st, zs):
    p = st.real ** 2 + st.imag ** 2
    return p @ zs

def bloch(bk, st, q, n):
    """Bloch vector (<X>,<Y>,<Z>) of qubit q (reduced state) -> (B,3)"""
    B = st.shape[0]; s = st.reshape(B, 2 ** q, 2, 2 ** (n - q - 1))
    a0, a1 = s[:, :, 0, :], s[:, :, 1, :]
    r01 = (a0 * a1.conj()).sum((1, 2)) if bk.torch else (a0 * a1.conj()).sum((1, 2))   # rho_01 = sum psi_0 psi_1*
    pz = (abs(a0) ** 2).sum((1, 2)) - (abs(a1) ** 2).sum((1, 2))
    return bk.stack([2 * r01.real, -2 * r01.imag, pz], 1)

def n_params(n, L):
    return 3 * n * L
''', "s12_quantum.py": r'''# s12_quantum.py - S12 quantum part for ONE dataset (CPU, exact state-vector simulation; rules: S12_preregistration.txt).
# Poster pipeline: image -> preprocessing & patch extraction -> encoding (PCA + angle = main; amplitude / basis = poster's
# other schemes) -> VQC (Rx Ry Rz + CNOT ring) x L -> measure <Z_i> -> classical dense layer -> softmax.
#   "pure"  : VQC on 4x4 patch statistics of the image (poster "VQC (Ours)")
#   "hybrid": VQC on frozen ConvNeXt features (poster "Hybrid CNN+VQC")
# Fair controls on IDENTICAL inputs: logistic regression; MLP with the SAME number of trainable parameters as the VQC head.
# Literature controls: no-entanglement VQC (Bowles et al. 2024), data re-uploading (Perez-Salinas et al. 2020),
# quantum-kernel SVM (Havlicek et al. 2019) vs classical RBF-kernel SVM.
import os, sys, json, time, argparse
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import qsim as Q

ap = argparse.ArgumentParser()
ap.add_argument("--ds", choices=["cmmd", "cbis"], required=True)
ap.add_argument("--data", required=True); ap.add_argument("--feats", required=True); ap.add_argument("--out", required=True)
ap.add_argument("--seeds", default="0,1,2,3,4"); ap.add_argument("--epochs", type=int, default=60)
ap.add_argument("--budget_h", type=float, default=3.0)        # secondary configurations are skipped once this is used up
ap.add_argument("--smoke", action="store_true")
a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
torch.set_num_threads(2); T0 = time.time()
def log(*m): print(f"[S12-quantum-{a.ds}]", *m, flush=True)
bk = Q.Backend(torch)

def val_mask(patients, frac=0.15, seed=0):
    ups = sorted(set(patients)); rng = np.random.default_rng(seed)
    val = set(rng.choice(ups, int(round(frac * len(ups))), replace=False).tolist())
    return np.array([p in val for p in patients])
d = np.load(os.path.join(a.data, "train_cache.npz")); vm = val_mask(d["patient"].astype(str))
X = {"train": d["x"][~vm], "val": d["x"][vm], "test": np.load(os.path.join(a.data, "test_cache.npz"))["x"]}
C = np.load(a.feats); Y = {k: C[f"y_{k}"].astype(np.int64) for k in ("train", "val", "test")}
if a.smoke:
    X = {k: v[:len(Y[k])] for k, v in X.items()}
assert all(len(X[k]) == len(Y[k]) for k in X), {k: (len(X[k]), len(Y[k])) for k in X}

def patch_feats(U8):
    """poster step 2: resize 256 -> 64, per-image min-max, 4x4 grid of 16x16 patches, per patch (mean, std, high-pass energy)"""
    x = U8.astype(np.float32).reshape(len(U8), 64, 4, 64, 4).mean((2, 4))
    mn = x.min((1, 2), keepdims=True); mx = x.max((1, 2), keepdims=True); x = (x - mn) / (mx - mn + 1e-8)
    pad = np.pad(x, ((0, 0), (1, 1), (1, 1)), mode="edge")
    box = sum(pad[:, i:i + 64, j:j + 64] for i in range(3) for j in range(3)) / 9.0; hp = (x - box) ** 2
    P = lambda z: z.reshape(len(z), 4, 16, 4, 16).transpose(0, 1, 3, 2, 4).reshape(len(z), 16, 256)
    return np.concatenate([P(x).mean(2), P(x).std(2), P(hp).mean(2)], 1)          # (N, 48)
RAW = {"pure": {k: patch_feats(v) for k, v in X.items()}, "hybrid": {k: C[f"f_{k}"] for k in X}}

def pca(kind, n):
    """standardise (train-fit stats) -> PCA to n components (fitted on train-fit) -> z-score"""
    R = RAW[kind]; mu, sd = R["train"].mean(0), R["train"].std(0) + 1e-8
    Z = {k: (v - mu) / sd for k, v in R.items()}
    _, S, Vt = np.linalg.svd(Z["train"], full_matrices=False); V = Vt[:n].T
    P = {k: v @ V for k, v in Z.items()}; m2, s2 = P["train"].mean(0), P["train"].std(0) + 1e-8
    return {k: ((v - m2) / s2).astype(np.float32) for k, v in P.items()}, float((S[:n] ** 2).sum() / (S ** 2).sum())
def encode(kind, n, enc):
    """returns (classical input Z, quantum input) ; angle: 2*arctan(z) | amplitude: 2^n PCA numbers, unit norm, on n qubits |
    basis: sign(z) bits -> computational basis state"""
    if enc == "amp":
        Z, ev = pca(kind, 2 ** n)
        # fix 2026-10-23: patch features have only 48 numbers < 2^6 = 64 amplitudes -> zero-pad (standard amplitude encoding)
        pad = lambda v: np.pad(v, ((0, 0), (0, 2 ** n - v.shape[1]))) if v.shape[1] < 2 ** n else v
        A = {k: pad(v) / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-8) for k, v in Z.items()}
        return Z, {k: v.astype(np.float32) for k, v in A.items()}, ev
    Z, ev = pca(kind, n)
    if enc == "basis":
        w = 2 ** np.arange(n - 1, -1, -1)
        return Z, {k: ((v > 0).astype(np.int64) * w).sum(1) for k, v in Z.items()}, ev
    return Z, {k: (2 * np.arctan(v)).astype(np.float32) for k, v in Z.items()}, ev

class VQCHead(nn.Module):
    def __init__(self, n, L, enc="angle", reupload=False, entangle=True):
        super().__init__(); self.n, self.enc, self.reupload, self.entangle = n, enc, reupload, entangle
        self.W = nn.Parameter(0.1 * torch.randn(L, n, 3))                          # near-identity start (pre-stated)
        self.dense = nn.Linear(n, 2); self.perm = Q.cnot_ring_perm(n); self.zs = torch.tensor(Q.zsign(n))
    def state(self, xin):
        B = xin.shape[0]
        if self.enc == "amp":
            init = torch.complex(xin, torch.zeros_like(xin)); ang = torch.zeros(B, self.n)
        elif self.enc == "basis":
            re = torch.zeros(B, 2 ** self.n); re[torch.arange(B), xin] = 1.0
            init = torch.complex(re, torch.zeros_like(re)); ang = torch.zeros(B, self.n)
        else:
            init, ang = None, xin
        return Q.vqc_state(bk, ang, self.W, self.perm, init=init, reupload=self.reupload, entangle=self.entangle)
    def forward(self, xin):
        return self.dense(Q.expz(bk, self.state(xin), self.zs))
class MLPHead(nn.Module):
    def __init__(self, n_in, h):
        super().__init__(); self.net = nn.Sequential(nn.Linear(n_in, h), nn.Tanh(), nn.Linear(h, 2))
    def forward(self, z):
        return self.net(z)
nparams = lambda m: sum(p.numel() for p in m.parameters() if p.requires_grad)

def fit(model, Xtr, ytr, seed, epochs, bs=64, lr=0.01):
    w = torch.tensor([len(ytr) / (2 * max(1, (ytr == c).sum())) for c in (0, 1)], dtype=torch.float32)
    opt = torch.optim.Adam(model.parameters(), lr=lr); Xt, Yt = torch.from_numpy(Xtr), torch.from_numpy(ytr)
    g = torch.Generator().manual_seed(seed); hist = []
    for ep in range(epochs):
        perm = torch.randperm(len(Yt), generator=g); tl = 0.0
        for i in range(0, len(Yt), bs):
            idx = perm[i:i + bs]; loss = F.cross_entropy(model(Xt[idx]), Yt[idx], weight=w)
            opt.zero_grad(); loss.backward(); opt.step(); tl += float(loss) * len(idx)
        hist.append(tl / len(Yt))
    return model, hist
@torch.no_grad()
def proba(model, Xn):
    return torch.softmax(model(torch.from_numpy(Xn)), 1)[:, 1].numpy()
def auc(s, y):
    from scipy.stats import rankdata
    r = rankdata(s); n1, n0 = (y == 1).sum(), (y == 0).sum(); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def save(name, pv, pt, **kw):
    np.savez(os.path.join(a.out, f"pred_{a.ds}_{name}.npz"), val=pv, test=pt, y_val=Y["val"], y_test=Y["test"], **kw)

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
# (feature set, qubits, layers, encoding, re-upload, entangle, role) - in priority order
CONFIGS = [("hybrid", 8, 4, "angle", False, True, "main"), ("pure", 8, 4, "angle", False, True, "main"),
           ("hybrid", 8, 4, "angle", False, False, "noent"), ("pure", 8, 4, "angle", False, False, "noent"),
           ("hybrid", 8, 4, "angle", True, True, "reupload"), ("pure", 8, 4, "angle", True, True, "reupload"),
           ("hybrid", 6, 4, "amp", False, True, "encoding"), ("pure", 6, 4, "amp", False, True, "encoding"),
           ("hybrid", 8, 4, "basis", False, True, "encoding"),
           ("hybrid", 4, 2, "angle", False, True, "size"), ("hybrid", 4, 4, "angle", False, True, "size"),
           ("hybrid", 8, 2, "angle", False, True, "size")]
if a.smoke:
    CONFIGS = [CONFIGS[0], CONFIGS[1], CONFIGS[4], CONFIGS[6], CONFIGS[8]]; a.epochs = 2
SEEDS = [int(s) for s in a.seeds.split(",")]; META = {}
mpath = os.path.join(a.out, f"quantum_meta_{a.ds}.json")
if os.path.exists(mpath):
    META = json.load(open(mpath))

# ---------------- quantum kernel SVM vs classical RBF SVM (fixed feature map, no training of the circuit) ----------------
for kind in ("hybrid", "pure"):
    Z, ANG, _ = encode(kind, 8, "angle")
    with torch.no_grad():
        S = {k: Q.feature_map_state(Q.Backend(np), v.astype(np.float64), reps=2) for k, v in ANG.items()}
    Ktr = np.abs(S["train"].conj() @ S["train"].T) ** 2
    qs = SVC(kernel="precomputed", C=1.0, class_weight="balanced").fit(Ktr, Y["train"])
    sig = lambda z: 1 / (1 + np.exp(-z))
    pv = sig(qs.decision_function(np.abs(S["val"].conj() @ S["train"].T) ** 2))
    pt = sig(qs.decision_function(np.abs(S["test"].conj() @ S["train"].T) ** 2)); save(f"{kind}_qkernel_s0", pv, pt)
    rb = SVC(kernel="rbf", C=1.0, gamma="scale", class_weight="balanced").fit(Z["train"], Y["train"])
    save(f"{kind}_rbfsvm_s0", sig(rb.decision_function(Z["val"])), sig(rb.decision_function(Z["test"])))
    log(f"{kind}: quantum-kernel SVM test AUC {auc(pt, Y['test']):.3f} | RBF SVM test AUC {auc(sig(rb.decision_function(Z['test'])), Y['test']):.3f}")

for kind, n, L, enc, reup, ent, role in CONFIGS:
    if role != "main" and (time.time() - T0) / 3600 > a.budget_h:
        log(f"time budget {a.budget_h} h used -> {role} config {kind} {n}q {L}l {enc} skipped"); continue
    cfg = f"{kind}_q{n}_l{L}_{enc}" + ("_reup" if reup else "") + ("" if ent else "_noent")
    Z, QIN, ev = encode(kind, n, enc)
    vq_params = Q.n_params(n, L) + 2 * n + 2; n_in = Z["train"].shape[1]
    h = max(1, round((vq_params - 2) / (n_in + 3)))                                  # MLP n_in -> h -> 2 with ~ the same parameters
    META[cfg] = {**META.get(cfg, {}), "kind": kind, "qubits": n, "layers": L, "encoding": enc, "reupload": reup, "entangle": ent, "role": role,
                 "pca_components": n_in, "pca_explained_variance": ev, "mlp_hidden": h}
    lr_ = LogisticRegression(class_weight="balanced", max_iter=5000).fit(Z["train"], Y["train"])
    save(f"{cfg}_LR_s0", lr_.predict_proba(Z["val"])[:, 1], lr_.predict_proba(Z["test"])[:, 1])
    for seed in SEEDS:
        for head in ("VQC", "MLP"):
            f = os.path.join(a.out, f"pred_{a.ds}_{cfg}_{head}_s{seed}.npz")
            if os.path.exists(f):
                continue
            torch.manual_seed(seed); t0 = time.time()
            m = VQCHead(n, L, enc, reup, ent) if head == "VQC" else MLPHead(n_in, h)
            Xin = QIN if head == "VQC" else Z
            m, hist = fit(m, Xin["train"], Y["train"], seed, a.epochs)
            pv, pt = proba(m, Xin["val"]), proba(m, Xin["test"]); META[cfg][f"params_{head}"] = nparams(m)
            extra = {}
            if head == "VQC" and role == "main" and seed == SEEDS[0]:
                with torch.no_grad():
                    st = m.state(torch.from_numpy(Xin["test"]))
                    extra = {"zvec_test": Q.expz(bk, st, m.zs).numpy(), "pca_test": Z["test"],
                             "bloch_test": np.stack([Q.bloch(bk, st, q, n).numpy() for q in range(n)], 1), "W": m.W.detach().numpy()}
            save(f"{cfg}_{head}_s{seed}", pv, pt, loss=np.array(hist), **extra)
            log(f"{cfg} {head} seed {seed}: {nparams(m)} params | val AUC {auc(pv, Y['val']):.3f} | test AUC {auc(pt, Y['test']):.3f} | "
                f"final loss {hist[-1]:.4f} | {time.time() - t0:.0f} s | elapsed {(time.time() - T0) / 60:.0f} min")
    json.dump(META, open(mpath, "w"), indent=1)
log(f"done in {(time.time() - T0) / 60:.0f} min")
''', "s12_stats.py": r'''# s12_stats.py - S12 endpoints, poster-style results tables and figures for CMMD and CBIS-DDSM (CPU).
# usage: python s12_stats.py <S12 output folder> <cmmd_data folder>
import os, sys, glob, json
import numpy as np
from scipy.stats import rankdata
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

D, DATA = sys.argv[1], sys.argv[2]
PAL = ["#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e34948", "#c79a1e", "#3aa6b9", "#8a8a8a", "#b0559b", "#4d7f2e"]
plt.rcParams.update({"figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.spines.top": False, "axes.spines.right": False,
                     "axes.grid": True, "grid.color": "#e4e3df", "font.size": 9, "savefig.dpi": 170, "savefig.bbox": "tight"})
DSN = {"cmmd": "CMMD", "cbis": "CBIS-DDSM"}
def auc(p, y):
    r = rankdata(p); n1 = (y == 1).sum(); n0 = (y == 0).sum(); return float((r[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))
def runs(ds, pattern):
    return [dict(np.load(f)) for f in sorted(glob.glob(os.path.join(D, f"pred_{ds}_{pattern}")))]
ens = lambda rs, s: np.mean([r[s] for r in rs], 0)
def metrics(p, y):
    yh = (p >= 0.5).astype(int); tp = ((yh == 1) & (y == 1)).sum(); fp = ((yh == 1) & (y == 0)).sum(); fn = ((yh == 0) & (y == 1)).sum()
    pr = tp / max(1, tp + fp); rc = tp / max(1, tp + fn)
    return {"accuracy": float((yh == y).mean()), "precision": float(pr), "recall": float(rc), "f1": float(2 * pr * rc / max(1e-9, pr + rc))}
M8 = "q8_l4_angle"
ARMS = {  # label -> pattern; poster box 9 models + fair controls + literature controls
    "Classical CNN (small, scratch)": "cnn_smallcnn_s*.npz", "ResNet50": "cnn_resnet50_s*.npz",
    "EfficientNet-B0": "cnn_efficientnet_b0_s*.npz", "ConvNeXt-Tiny": "cnn_convnext_tiny_s*.npz",
    "CNN-PCA8 + logistic regression": f"hybrid_{M8}_LR_s*.npz", "CNN-PCA8 + MLP (matched size)": f"hybrid_{M8}_MLP_s*.npz",
    "CNN-PCA8 + RBF-kernel SVM": "hybrid_rbfsvm_s*.npz", "CNN-PCA8 + quantum-kernel SVM": "hybrid_qkernel_s*.npz",
    "Hybrid CNN+VQC": f"hybrid_{M8}_VQC_s*.npz", "Hybrid CNN+VQC, no entanglement": f"hybrid_{M8}_noent_VQC_s*.npz",
    "Hybrid CNN+VQC, data re-uploading": f"hybrid_{M8}_reup_VQC_s*.npz",
    "patch-PCA8 + logistic regression": f"pure_{M8}_LR_s*.npz", "patch-PCA8 + MLP (matched size)": f"pure_{M8}_MLP_s*.npz",
    "patch-PCA8 + RBF-kernel SVM": "pure_rbfsvm_s*.npz", "patch-PCA8 + quantum-kernel SVM": "pure_qkernel_s*.npz",
    "VQC on patches": f"pure_{M8}_VQC_s*.npz", "VQC on patches, no entanglement": f"pure_{M8}_noent_VQC_s*.npz",
    "VQC on patches, data re-uploading": f"pure_{M8}_reup_VQC_s*.npz"}
L = ["S12 REPORT - quantum (VQC) classifier as in the supervisor's poster, benign vs malignant, CMMD + CBIS-DDSM",
     "(rules: S12_preregistration.txt)", "=" * 120,
     "Simulation: exact state-vector simulator (verified against dense matrices; PennyLane cross-check below). No quantum hardware.",
     "8 qubits x 4 layers = 96 circuit + 18 dense parameters; VQC = 5 seeds, MLP = 5 seeds, CNNs = 3 seeds, LR / SVMs deterministic."]
pc = os.path.join(D, "pennylane_check.json")
if os.path.exists(pc):
    j = json.load(open(pc)); L.append(f"PennyLane {j['pennylane']} cross-check: max |<Z> difference| {j['max_abs_diff']:.1e}")
rng = np.random.default_rng(0)
def boot(pa, pb, y, alpha, n=2000):
    i0, i1 = np.where(y == 0)[0], np.where(y == 1)[0]; d = []
    for _ in range(n):
        b = np.concatenate([rng.choice(i0, len(i0)), rng.choice(i1, len(i1))]); d.append(auc(pa[b], y[b]) - auc(pb[b], y[b]))
    return float(np.quantile(d, alpha / 2)), float(np.quantile(d, 1 - alpha / 2))
OUT = {}

def do_dataset(ds):
    R = {k: runs(ds, v) for k, v in ARMS.items()}; O = {"arms": {}, "endpoints": {}}
    META = json.load(open(os.path.join(D, f"quantum_meta_{ds}.json"))) if os.path.exists(os.path.join(D, f"quantum_meta_{ds}.json")) else {}
    miss = [k for k, v in R.items() if not v]
    L.extend(["", "#" * 120, f"DATASET: {DSN[ds]}" + (f"   (missing arms: {miss})" if miss else "   (all arms present)"), "#" * 120,
              f"RESULTS TABLE - {DSN[ds]} TEST (ensemble = mean probability over seeds; threshold 0.5 for accuracy / precision / recall / F1)",
              f"  {'model':40s} {'params':>9s} {'seeds':>5s} {'AUC':>6s} {'seed AUC mean+-SD':>18s} {'acc':>6s} {'prec':>6s} {'rec':>6s} {'F1':>6s}"])
    par = {"Hybrid CNN+VQC": META.get(f"hybrid_{M8}", {}).get("params_VQC"), "CNN-PCA8 + MLP (matched size)": META.get(f"hybrid_{M8}", {}).get("params_MLP"),
           "VQC on patches": META.get(f"pure_{M8}", {}).get("params_VQC"), "patch-PCA8 + MLP (matched size)": META.get(f"pure_{M8}", {}).get("params_MLP"),
           "CNN-PCA8 + logistic regression": 9, "patch-PCA8 + logistic regression": 9}
    for k, rs in R.items():
        if not rs:
            continue
        y = rs[0]["y_test"]; p = ens(rs, "test"); s = [auc(r["test"], y) for r in rs]; m = metrics(p, y)
        O["arms"][k] = {"auc": auc(p, y), "seed_aucs": s, "val_auc": auc(ens(rs, "val"), rs[0]["y_val"]), **m}
        sd = f"{np.mean(s):.3f}+-{np.std(s, ddof=1):.3f}" if len(s) > 1 else f"{s[0]:.3f}"
        pp = par.get(k) or ("full net" if k in ("Classical CNN (small, scratch)", "ResNet50", "EfficientNet-B0", "ConvNeXt-Tiny") else "-")
        L.append(f"  {k:40s} {str(pp):>9s} "
                 f"{len(rs):>5d} {auc(p, y):6.3f} {sd:>18s} {m['accuracy']:6.3f} {m['precision']:6.3f} {m['recall']:6.3f} {m['f1']:6.3f}")
    cf = os.path.join(D, f"cnn_features_{ds}.npz")
    if os.path.exists(cf):
        C = np.load(cf); L.append(f"  (frozen ConvNeXt that supplies the hybrid features, its own head: test AUC {auc(C['p_test'], C['y_test']):.3f})")
    def endpoint(name, A, B, alpha, primary):
        if not (R.get(A) and R.get(B)):
            L.append(f"  {name}: arm missing"); return None
        y = R[A][0]["y_test"]; pa, pb = ens(R[A], "test"), ens(R[B], "test"); dd = auc(pa, y) - auc(pb, y); lo, hi = boot(pa, pb, y, alpha)
        eq = (lo > -0.02) and (hi < 0.02)
        verdict = ("PASS" if lo > 0 else "fail") if primary else ("CI excludes 0" if (lo > 0 or hi < 0) else "CI includes 0")
        L.append(f"  {name:4s} AUC[{A}] - AUC[{B}] = {dd:+.4f}  CI{100 * (1 - alpha):.2f}% {lo:+.4f}..{hi:+.4f} -> {verdict}"
                 + (" | within +-0.02 (practically equivalent)" if eq else ""))
        return {"diff": dd, "ci": [lo, hi], "verdict": verdict, "passed": bool(lo > 0), "equivalent_0.02": bool(eq)}
    L.extend(["", f"PRIMARY ENDPOINTS on {DSN[ds]} (98.33 % CI = Bonferroni over Q1-Q3; PASS = CI lower bound > 0)"])
    for q, A, B in (("Q1", "Hybrid CNN+VQC", "CNN-PCA8 + MLP (matched size)"), ("Q2", "VQC on patches", "patch-PCA8 + MLP (matched size)"),
                    ("Q3", "Hybrid CNN+VQC", "ConvNeXt-Tiny")):
        O["endpoints"][q] = endpoint(q, A, B, 0.05 / 3, True)
    L.extend(["", "SECONDARY (95 % CI, descriptive)"])
    for nm, A, B in (("S1", "Hybrid CNN+VQC", "CNN-PCA8 + logistic regression"), ("S2", "VQC on patches", "patch-PCA8 + logistic regression"),
                     ("S3", "Hybrid CNN+VQC", "ResNet50"), ("S4", "Hybrid CNN+VQC", "EfficientNet-B0"),
                     ("S5", "Hybrid CNN+VQC", "Classical CNN (small, scratch)"), ("S6", "Hybrid CNN+VQC", "VQC on patches"),
                     ("S7", "Hybrid CNN+VQC", "Hybrid CNN+VQC, no entanglement"), ("S8", "VQC on patches", "VQC on patches, no entanglement"),
                     ("S9", "Hybrid CNN+VQC, data re-uploading", "CNN-PCA8 + MLP (matched size)"),
                     ("S10", "VQC on patches, data re-uploading", "patch-PCA8 + MLP (matched size)"),
                     ("S11", "CNN-PCA8 + quantum-kernel SVM", "CNN-PCA8 + RBF-kernel SVM"),
                     ("S12", "patch-PCA8 + quantum-kernel SVM", "patch-PCA8 + RBF-kernel SVM")):
        O["endpoints"][nm] = endpoint(nm, A, B, 0.05, False)
    L.extend(["", "ENCODINGS AND CIRCUIT SIZE (descriptive): each VQC vs its size-matched MLP and logistic regression on identical inputs"])
    for cfg, mt in META.items():
        v, m_, lr_ = runs(ds, f"{cfg}_VQC_s*.npz"), runs(ds, f"{cfg}_MLP_s*.npz"), runs(ds, f"{cfg}_LR_s*.npz")
        if v and m_ and lr_:
            y = v[0]["y_test"]
            L.append(f"  {cfg:28s} ({mt['role']:8s}; {mt['pca_components']:2d} PCA numbers keep {100 * mt['pca_explained_variance']:4.1f} % variance): "
                     f"VQC {auc(ens(v, 'test'), y):.3f} ({mt.get('params_VQC')} p) | MLP {auc(ens(m_, 'test'), y):.3f} ({mt.get('params_MLP')} p) | "
                     f"LR {auc(ens(lr_, 'test'), y):.3f}")
    # ---- figures: box 4 (Bloch), box 7 (feature space), box 9 (ROC + AUC bars) ----
    vq = runs(ds, f"hybrid_{M8}_VQC_s0.npz")
    if vq and "bloch_test" in vq[0]:
        v = vq[0]; yb = v["y_test"]; B = v["bloch_test"]
        fig = plt.figure(figsize=(11, 3.8))
        for j, q in enumerate((0, 1, 2)):
            a_ = fig.add_subplot(1, 3, j + 1, projection="3d"); u, w = np.mgrid[0:2 * np.pi:30j, 0:np.pi:15j]
            a_.plot_wireframe(np.cos(u) * np.sin(w), np.sin(u) * np.sin(w), np.cos(w), color="#dddddd", lw=0.3)
            for c, col, lab in ((0, PAL[0], "benign"), (1, PAL[4], "malignant")):
                P = B[yb == c, q]; a_.scatter(P[:, 0], P[:, 1], P[:, 2], s=5, color=col, label=lab, alpha=0.6)
            sep = np.linalg.norm(B[yb == 1, q].mean(0) - B[yb == 0, q].mean(0))
            a_.set_title(f"{DSN[ds]} qubit {q} Bloch vectors\n(class-mean distance {sep:.2f})", fontsize=8); a_.set_box_aspect((1, 1, 1))
            if j == 0:
                a_.legend(fontsize=7, loc="upper left")
        fig.savefig(f"{D}/S12_{ds}_box4_bloch.png"); plt.close(fig)
        L.extend(["", "BLOCH VECTORS (hybrid VQC, seed 0, test images): distance between class-mean Bloch vectors per qubit (max possible 2)",
                  "  " + " | ".join(f"q{q} {np.linalg.norm(B[yb == 1, q].mean(0) - B[yb == 0, q].mean(0)):.3f}" for q in range(B.shape[1]))])
        from sklearn.metrics import silhouette_score
        from sklearn.manifold import TSNE
        fig, ax = plt.subplots(1, 3, figsize=(12, 3.7)); sil = {}
        for a_, (nm, F_) in zip(ax[:2], (("classical PCA input (8 numbers)", v["pca_test"]), ("quantum features <Z> (8 numbers)", v["zvec_test"]))):
            Fc = F_ - F_.mean(0); _, _, Vt_ = np.linalg.svd(Fc, full_matrices=False); P2 = Fc @ Vt_[:2].T; sil[nm] = float(silhouette_score(F_, yb))
            for c, col, lab, mk in ((0, PAL[0], "benign", "o"), (1, PAL[4], "malignant", "^")):
                a_.scatter(P2[yb == c, 0], P2[yb == c, 1], s=9, color=col, marker=mk, label=lab, alpha=0.7)
            a_.set_title(f"{DSN[ds]}: PCA of {nm}\nsilhouette {sil[nm]:+.3f}", loc="left", fontsize=8.5); a_.set_xlabel("PC 1"); a_.set_ylabel("PC 2")
        ax[0].legend(frameon=False, fontsize=7.5)
        T = TSNE(2, random_state=0, perplexity=30, init="pca").fit_transform(v["zvec_test"])
        for c, col, lab, mk in ((0, PAL[0], "benign", "o"), (1, PAL[4], "malignant", "^")):
            ax[2].scatter(T[yb == c, 0], T[yb == c, 1], s=9, color=col, marker=mk, label=lab, alpha=0.7)
        ax[2].set_title(f"{DSN[ds]}: t-SNE of quantum features <Z>", loc="left", fontsize=8.5)
        fig.savefig(f"{D}/S12_{ds}_box7_feature_space.png"); plt.close(fig)
        L.extend(["", "FEATURE-SPACE SEPARATION (silhouette score, test images; higher = classes more separated; -1..1)"]
                 + [f"  {k:34s}: {s:+.3f}" for k, s in sil.items()]); O["silhouette"] = sil
    names = [k for k in ARMS if k in O["arms"]]
    if names:
        fig, ax = plt.subplots(1, 2, figsize=(14, 5.2), gridspec_kw={"width_ratios": [1, 1.1]}); fig.subplots_adjust(wspace=0.9)
        KEY = ["Classical CNN (small, scratch)", "ResNet50", "EfficientNet-B0", "ConvNeXt-Tiny", "CNN-PCA8 + MLP (matched size)",
               "Hybrid CNN+VQC", "patch-PCA8 + MLP (matched size)", "VQC on patches"]
        for j, k in enumerate([k for k in KEY if k in names]):
            rs = R[k]; y = rs[0]["y_test"]; p = ens(rs, "test"); o = np.argsort(-p)
            tp = np.cumsum(y[o] == 1) / (y == 1).sum(); fp = np.cumsum(y[o] == 0) / (y == 0).sum(); qk = "VQC" in k or "quantum" in k
            ax[0].plot(np.r_[0, fp], np.r_[0, tp], color=PAL[j % len(PAL)], lw=1.8 if qk else 1.0, ls="-" if qk else "--",
                       label=f"{k} ({O['arms'][k]['auc']:.3f})")
        ax[0].plot([0, 1], [0, 1], color="#999", lw=0.8); ax[0].set_xlabel("false-positive rate"); ax[0].set_ylabel("true-positive rate")
        ax[0].set_title(f"ROC, {DSN[ds]} test (ensembles)", loc="left"); ax[0].legend(fontsize=7, frameon=False, loc="upper left", bbox_to_anchor=(0, -0.13), ncol=2)
        au = [O["arms"][k]["auc"] for k in names]
        ax[1].barh(range(len(names)), au, color=[PAL[4] if ("VQC" in k or "quantum" in k) else PAL[0] for k in names])
        ax[1].set_yticks(range(len(names)), names, fontsize=6.5); ax[1].set_xlim(0.45, max(au) + 0.06); ax[1].set_xlabel("test AUC")
        ax[1].invert_yaxis(); ax[1].set_title("AUC by model (red = quantum)", loc="left")
        for i, v_ in enumerate(au):
            ax[1].text(v_ + 0.003, i, f"{v_:.3f}", va="center", fontsize=6.5)
        fig.savefig(f"{D}/S12_{ds}_box9_results.png"); plt.close(fig)
    O["meta"] = META
    return O

for ds in ("cmmd", "cbis"):
    OUT[ds] = do_dataset(ds)
L.extend(["", "=" * 120, "OVERALL VERDICT (pre-registered): a question is SUPPORTED only if it PASSES on BOTH datasets"])
for q, txt in (("Q1", "Hybrid CNN+VQC beats the same-size classical head on the same CNN features"),
               ("Q2", "VQC on patches beats the same-size classical head on the same patch features"),
               ("Q3", "Hybrid CNN+VQC beats a fully fine-tuned ConvNeXt-Tiny")):
    r = [OUT[ds]["endpoints"].get(q) for ds in ("cmmd", "cbis")]
    st = ["missing" if x is None else ("PASS" if x["passed"] else "fail") for x in r]
    L.append(f"  {q}: {txt}: CMMD {st[0]} | CBIS-DDSM {st[1]} -> {'SUPPORTED' if st == ['PASS', 'PASS'] else 'NOT supported'}")

# ---- box 2 (patches) and box 5 (circuit), once ----
X = np.load(os.path.join(DATA, "test_cache.npz")); xt, yt = X["x"], X["y"]
fig, ax = plt.subplots(1, 3, figsize=(10, 3.4)); i0 = int(np.where(yt == 0)[0][0]); i1 = int(np.where(yt == 1)[0][0])
for a_, i, t in ((ax[0], i0, "benign"), (ax[1], i1, "malignant")):
    a_.imshow(xt[i], cmap="gray"); a_.set_title(f"CMMD test image ({t})", loc="left")
    for k in range(1, 4):
        a_.axhline(64 * k, color="#eb6834", lw=0.8); a_.axvline(64 * k, color="#eb6834", lw=0.8)
    a_.set_xticks([]); a_.set_yticks([]); a_.grid(False)
small = xt[i1].astype(np.float32).reshape(64, 4, 64, 4).mean((1, 3)); ax[2].imshow(small, cmap="gray"); ax[2].grid(False)
ax[2].set_title("64x64 input, 4x4 patches -> 48 numbers -> PCA 8", loc="left"); ax[2].set_xticks([]); ax[2].set_yticks([])
fig.savefig(f"{D}/S12_box2_patches.png"); plt.close(fig)
fig, ax = plt.subplots(figsize=(10, 3.3)); n = 8; ax.set_xlim(-0.5, 11); ax.set_ylim(-0.8, n); ax.axis("off")
for q in range(n):
    ax.plot([0.3, 10.6], [n - 1 - q] * 2, color="#555", lw=0.8); ax.text(-0.3, n - 1 - q, f"q{q} |0>", va="center", ha="right", fontsize=8)
    ax.add_patch(plt.Rectangle((0.5, n - 1.35 - q), 0.9, 0.7, color="#d9e8fb")); ax.text(0.95, n - 1 - q, "RY(x)", ha="center", va="center", fontsize=6.5)
for l in range(4):
    x0 = 2 + 2.1 * l
    for q in range(n):
        ax.add_patch(plt.Rectangle((x0, n - 1.35 - q), 1.0, 0.7, color="#fde4d8")); ax.text(x0 + 0.5, n - 1 - q, "Rx Ry Rz", ha="center", va="center", fontsize=6)
    for q in range(n):
        t = (q + 1) % n; xc = x0 + 1.2 + 0.08 * q
        ax.plot([xc, xc], [n - 1 - q, n - 1 - t], color="#1baf7a", lw=0.8); ax.plot(xc, n - 1 - q, "o", color="#1baf7a", ms=3)
    ax.text(x0 + 0.8, -0.6, f"layer {l + 1}", ha="center", fontsize=8)
for q in range(n):
    ax.text(10.75, n - 1 - q, "<Z>", va="center", fontsize=7)
ax.set_title("Main VQC: angle encoding, 4 layers of trainable Rx-Ry-Rz + CNOT ring, <Z> readout -> dense -> softmax", loc="left")
fig.savefig(f"{D}/S12_box5_circuit.png"); plt.close(fig)

txt = "\n".join(L); print(txt)
open(f"{D}/S12_report.txt", "w").write(txt + "\n"); json.dump(OUT, open(f"{D}/S12_results.json", "w"), indent=1, default=float)
'''}
for n, s in FILES.items():
    Path(f"{WD}/{n}").write_text(s)
def walk_find(name, root="/kaggle/input"):
    out = []
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("CMMD", "jpeg", "U24_synthesis", ".git", "ckpt", "train_logs", "logs")]
        if name in fns:
            out.append(os.path.join(dp, name))
    return sorted(out)
def need_dir(marker, parent, where):
    h = [os.path.dirname(p) for p in walk_find(marker) if os.path.basename(os.path.dirname(p)) == parent]
    if not h:
        raise SystemExit(f"!! {parent}/{marker} not found -> Add Input -> {where}")
    print(f"{parent:10s} -> {h[0]}"); return h[0]
S12_IN = need_dir("quantum_meta_cmmd.json", "S12", "the S12 — quantum notebook output")
DATA = {"cmmd": need_dir("cmmd_summary.json", "cmmd_data", "the S8 replication notebook output"),
        "cbis": need_dir("clean_summary.json", "clean_data", "the S8 replication or S2 baseline-audit output")}
shutil.rmtree(OUT, ignore_errors=True); shutil.copytree(S12_IN, OUT, copy_function=shutil.copyfile)   # writable copies
print("copied", len(glob.glob(f"{OUT}/pred_*.npz")), "existing prediction files")
for ds in ("cmmd", "cbis"):
    print(ds, "configurations already done:", list(json.load(open(f"{OUT}/quantum_meta_{ds}.json"))))
LOGS = f"{OUT}/logs"; os.makedirs(LOGS, exist_ok=True); RC = {}
def run_one(name, argv):
    env = os.environ.copy(); env["PYTHONUNBUFFERED"] = "1"; env["CUDA_VISIBLE_DEVICES"] = ""
    print(f">>> {name} started", flush=True)
    with open(f"{LOGS}/{name}.log", "w") as fh:
        p = subprocess.Popen([sys.executable, "-u"] + argv, cwd=WD, env=env, text=True, bufsize=1, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            fh.write(line); fh.flush(); print(line if line.startswith("[") else f"[{name}] {line}", end="", flush=True)
        RC[name] = p.wait()
    print(f"<<< {name} return code {RC[name]}" + ("" if RC[name] == 0 else f"  !! see {LOGS}/{name}.log"), flush=True)
print("Cell 1 done.")

In [ ]:
# ===================== CELL 2 — run the missing configurations (existing files are skipped) (~1 h) =====================
t0 = time.time()
ths = [threading.Thread(target=run_one, args=(f"quantum_complete_{ds}", ["s12_quantum.py", "--ds", ds, "--data", DATA[ds],
       "--feats", f"{OUT}/cnn_features_{ds}.npz", "--out", OUT, "--budget_h", "6"])) for ds in ("cmmd", "cbis")]
for t in ths: t.start()
for t in ths: t.join()
print(f"finished in {(time.time() - t0) / 60:.0f} min | return codes {RC}")
for ds in ("cmmd", "cbis"):
    print(ds, "configurations now:", list(json.load(open(f"{OUT}/quantum_meta_{ds}.json"))))

In [ ]:
# ===================== CELL 3 — rebuild the full S12 report + zip =====================
r = subprocess.run([sys.executable, "s12_stats.py", OUT, DATA["cmmd"]], cwd=WD, capture_output=True, text=True)
note = ("NOTE: S12 report rebuilt after S12b (2026-10-23). S12b ran ONLY the secondary configurations that had not run in S12 "
        "because of a code error (patch amplitude encoding: 48 numbers < 64 amplitudes; fixed by zero-padding). All other results "
        "are the original S12 files, unchanged.\n\n")
rep = f"{OUT}/S12_report.txt"
if os.path.exists(rep):
    body = open(rep).read(); open(rep, "w").write(note + body)        # read first, then write (fix 2026-10-23)
print(note + r.stdout[-40000:]); print(r.stderr[-3000:] if r.returncode else "")
json.dump(RC, open(f"{OUT}/return_codes_S12b.json", "w"), indent=1)
z = shutil.make_archive("/kaggle/working/S12_results_complete", "zip", OUT)
print("ZIP:", z, f"({os.path.getsize(z) / 1e6:.1f} MB) | return codes {RC}")